In [ ]:
!pip install transformers bitsandbytes accelerate datasets soundfile -q
!nvidia-smi

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 35.1 MB/s eta 0:00:00
Sun Jun 14 20:35:22 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   48C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                  

In [ ]:
!python scripts/extract_logits.py --format A --end 20

python3: can't open file '/content/scripts/extract_logits.py': [Errno 2] No such file or directory


In [ ]:
!python scripts/extract_logits.py --format both --resume

python3: can't open file '/content/scripts/extract_logits.py': [Errno 2] No such file or directory


In [ ]:
"""
Smoke test: 10-item end-to-end logit extraction on Qwen2-Audio-7B-Instruct (4-bit).

Tests both Format A (A/B tokens) and Format B (1/2 tokens).
Also runs Strategy 1 (model.generate) on the same 10 items and asserts
argmax agreement with Strategy 2 (model.forward logits).

Run on Colab after installing dependencies:
    !pip install transformers bitsandbytes accelerate datasets soundfile -q
    !python scripts/smoke_test.py

Results are printed to stdout and saved to results/smoke_test_results.json.
"""

import gc
import json
import os


# Must be set before torch is imported to take effect during weight loading.
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import torch
import numpy as np
from datasets import load_dataset, Audio
from transformers import AutoProcessor, Qwen2AudioForConditionalGeneration, BitsAndBytesConfig

# ---------------------------------------------------------------------------
# Hardcoded token IDs (verified 2026-06-02 on Qwen2-Audio-7B-Instruct)
# Format A: prompt ends with "Answer:"  → next token is ' A' or ' B'
# Format B: prompt ends with "Answer: " → next token is '1'  or '2'
# ---------------------------------------------------------------------------
TOKEN_A = 362   # ' A' (space-prefixed)
TOKEN_B = 425   # ' B' (space-prefixed)
TOKEN_1 = 16    # '1'  (no leading space; space is absorbed into prompt)
TOKEN_2 = 17    # '2'  (no leading space; space is absorbed into prompt)

MODEL_ID  = "Qwen/Qwen2-Audio-7B-Instruct"
DATASET   = "slprl/StressTest"
N_SMOKE   = 20
RESULTS_PATH = "results/smoke_test_results.json"

# ---------------------------------------------------------------------------
# Format A prompt builder
# ---------------------------------------------------------------------------
FORMAT_A_TEMPLATE = (
    "Out of the following answers, according to the speaker's stressed words, "
    "what is most likely the underlying intention of the speaker? "
    "A. {ans_a} B. {ans_b}"
)

# Suffix appended AFTER apply_chat_template so it lands inside the assistant
# turn — making the next predicted token the answer token itself.
FORMAT_A_SUFFIX = "Answer:"
FORMAT_B_SUFFIX = "Answer: "

def build_format_a_prompt(possible_answers):
    return FORMAT_A_TEMPLATE.format(
        ans_a=possible_answers[0],
        ans_b=possible_answers[1],
    )

def build_format_b_prompt(audio_lm_prompt):
    # Strip the dataset's trailing "Answer:" / "Answer: " — we'll re-append it
    # after the chat template wrapper so it sits inside the assistant turn.
    p = audio_lm_prompt.rstrip()
    if p.endswith("Answer:"):
        p = p[: -len("Answer:")].rstrip()
    return p

# ---------------------------------------------------------------------------
# Load model
# ---------------------------------------------------------------------------
def load_model():
    print(f"Loading processor from {MODEL_ID} ...")
    processor = AutoProcessor.from_pretrained(MODEL_ID)

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
    )

    gc.collect()
    torch.cuda.empty_cache()
    print(f"Loading model in 4-bit ...")
    model = Qwen2AudioForConditionalGeneration.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        device_map="auto",
        torch_dtype=torch.float16,
        low_cpu_mem_usage=True,
    )
    model.eval()
    return processor, model

# ---------------------------------------------------------------------------
# Strategy 2: direct forward pass — logits at final prompt token position
# ---------------------------------------------------------------------------
def strategy2_logits(processor, model, audio_array, sr, text_prompt, fmt):
    """
    Returns (p_first, p_second) 2-way softmax probabilities.
    For Format A: first=TOKEN_A, second=TOKEN_B
    For Format B: first=TOKEN_1, second=TOKEN_2
    """
    assert fmt in ("A", "B")
    tok_first = TOKEN_A if fmt == "A" else TOKEN_1
    tok_second = TOKEN_B if fmt == "A" else TOKEN_2
    suffix = FORMAT_A_SUFFIX if fmt == "A" else FORMAT_B_SUFFIX

    conversation = [
        {
            "role": "user",
            "content": [
                {"type": "audio", "audio_url": "__local__"},
                {"type": "text",  "text": text_prompt},
            ],
        }
    ]

    text = processor.apply_chat_template(
        conversation, add_generation_prompt=True, tokenize=False
    )
    # Place "Answer:" inside the assistant turn so the next predicted token
    # is the answer token itself, not the start of a free-form reply.
    text = text + suffix
    inputs = processor(
        text=text,
        audio=audio_array,
        return_tensors="pt",
        sampling_rate=sr,
    )
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    # logits shape: (batch, seq_len, vocab_size) — take last position
    logits_last = outputs.logits[0, -1, :]
    logit_first  = logits_last[tok_first].float().item()
    logit_second = logits_last[tok_second].float().item()

    # 2-way softmax
    max_l = max(logit_first, logit_second)
    exp_f = np.exp(logit_first  - max_l)
    exp_s = np.exp(logit_second - max_l)
    total = exp_f + exp_s
    return exp_f / total, exp_s / total

# ---------------------------------------------------------------------------
# Strategy 1: model.generate — returns the first new token ID (int)
#
# Greedy decoding is argmax of logits, so if Strategy 2 reads the right
# position and token IDs, generated[0, input_len] must equal
# argmax(logits[0, -1, :]). We compare raw IDs — no string parsing.
# ---------------------------------------------------------------------------
def strategy1_first_token_id(processor, model, audio_array, sr, text_prompt, fmt):
    assert fmt in ("A", "B")
    suffix = FORMAT_A_SUFFIX if fmt == "A" else FORMAT_B_SUFFIX

    conversation = [
        {
            "role": "user",
            "content": [
                {"type": "audio", "audio_url": "__local__"},
                {"type": "text",  "text": text_prompt},
            ],
        }
    ]

    text = processor.apply_chat_template(
        conversation, add_generation_prompt=True, tokenize=False
    )
    text = text + suffix
    inputs = processor(
        text=text,
        audio=audio_array,
        return_tensors="pt",
        sampling_rate=sr,
    )
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        generated = model.generate(
            **inputs,
            max_new_tokens=1,
            do_sample=False,
        )

    input_len = inputs["input_ids"].shape[1]
    return generated[0, input_len].item()

# ---------------------------------------------------------------------------
# Main
# ---------------------------------------------------------------------------
def main():
    processor, model = load_model()

    print(f"Loading dataset {DATASET} ...")
    ds = load_dataset(DATASET, split="test")
    ds = ds.cast_column("audio", Audio(sampling_rate=16000))
    items = [ds[i] for i in range(N_SMOKE)]

    results = []
    mismatches = []

    print(f"\nRunning {N_SMOKE}-item smoke test ...\n")
    header = f"{'i':>3}  {'label':>5}  {'FmtA argmax':>11}  {'FmtA p_A':>8}  {'FmtA p_B':>8}  {'FmtB argmax':>11}  {'FmtB p_1':>8}  {'FmtB p_2':>8}  {'gen_A':>6}  {'gen_B':>6}  {'match_A':>10}  {'match_B':>10}"
    print(header)
    print("-" * len(header))

    for i, item in enumerate(items):
        audio_array = np.array(item["audio"]["array"], dtype=np.float32)
        sr          = item["audio"]["sampling_rate"]
        label       = item["label"]           # 0 or 1
        answers     = item["possible_answers"]
        audio_lm    = item["audio_lm_prompt"]

        prompt_a = build_format_a_prompt(answers)
        prompt_b = build_format_b_prompt(audio_lm)

        # Strategy 2
        p_A, p_B = strategy2_logits(processor, model, audio_array, sr, prompt_a, "A")
        p_1, p_2 = strategy2_logits(processor, model, audio_array, sr, prompt_b, "B")

        argmax_a = 0 if p_A > p_B else 1   # index into possible_answers
        argmax_b = 0 if p_1 > p_2 else 1

        # Strategy 1: get first generated token ID (greedy = logit argmax)
        gen_a_tid = strategy1_first_token_id(processor, model, audio_array, sr, prompt_a, "A")
        gen_b_tid = strategy1_first_token_id(processor, model, audio_array, sr, prompt_b, "B")

        # Map token ID → answer index (0 or 1), or -1 if neither expected token
        gen_a_idx = 0 if gen_a_tid == TOKEN_A else (1 if gen_a_tid == TOKEN_B else -1)
        gen_b_idx = 0 if gen_b_tid == TOKEN_1 else (1 if gen_b_tid == TOKEN_2 else -1)

        # Decode token ID for display
        gen_a_str = processor.tokenizer.decode([gen_a_tid])
        gen_b_str = processor.tokenizer.decode([gen_b_tid])

        # FAIL if (a) Strategy 1 generated something other than an expected
        # answer token, or (b) Strategy 1 and Strategy 2 argmax disagree.
        # Previously match=None (unexpected token) was silently treated as
        # "not a failure" — that masked the chat-template position bug.
        match_a = (gen_a_idx != -1) and (argmax_a == gen_a_idx)
        match_b = (gen_b_idx != -1) and (argmax_b == gen_b_idx)

        if not match_a or not match_b:
            mismatches.append(i)

        row = {
            "idx": i,
            "label": int(label),
            "format_a": {
                "p_A": round(p_A, 4), "p_B": round(p_B, 4), "argmax": argmax_a,
                "gen_token_id": gen_a_tid, "gen_token": gen_a_str, "match": match_a,
            },
            "format_b": {
                "p_1": round(p_1, 4), "p_2": round(p_2, 4), "argmax": argmax_b,
                "gen_token_id": gen_b_tid, "gen_token": gen_b_str, "match": match_b,
            },
        }
        results.append(row)

        match_a_str = "OK" if match_a else f"FAIL({gen_a_str.strip()[:4]})"
        match_b_str = "OK" if match_b else f"FAIL({gen_b_str.strip()[:4]})"
        print(
            f"{i:>3}  {label:>5}  "
            f"{'AB'[argmax_a]:>11}  {p_A:>8.4f}  {p_B:>8.4f}  "
            f"{'12'[argmax_b]:>11}  {p_1:>8.4f}  {p_2:>8.4f}  "
            f"{gen_a_str.strip()[:6]:>6}  {gen_b_str.strip()[:6]:>6}  "
            f"{match_a_str:>10}  {match_b_str:>10}"
        )

    # Summary
    print(f"\n{'='*60}")
    if mismatches:
        print(f"SANITY CHECK FAILED on items: {mismatches}")
        print("Strategy 1 and Strategy 2 argmax disagree. Debug before proceeding.")
    else:
        print("SANITY CHECK PASSED: Strategy 1 and Strategy 2 argmax agree on all 10 items.")

    # Save results
    os.makedirs(os.path.dirname(RESULTS_PATH), exist_ok=True)
    with open(RESULTS_PATH, "w") as f:
        json.dump(results, f, indent=2)
    print(f"Results saved to {RESULTS_PATH}")

if __name__ == "__main__":
    main()


Loading processor from Qwen/Qwen2-Audio-7B-Instruct ...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/342 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/853 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/638k [00:00<?, ?B/s]

...

Loading dataset slprl/StressTest ...


README.md:   0%|          | 0.00/3.99k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/67.2M [00:00<?, ?B/s]

Generating test split:   0%|          | 0/218 [00:00<?, ? examples/s]


Running 20-item smoke test ...

  i  label  FmtA argmax  FmtA p_A  FmtA p_B  FmtB argmax  FmtB p_1  FmtB p_2   gen_A   gen_B     match_A     match_B
--------------------------------------------------------------------------------------------------------------------
  0      1            B    0.0038    0.9962            2    0.0210    0.9790       B       2          OK          OK
  1      0            B    0.0246    0.9754            2    0.0601    0.9399       B       2          OK          OK
  2      1            B    0.0023    0.9977            2    0.0450    0.9550       B       2          OK          OK
  3      1            A    0.8697    0.1303            1    0.7326    0.2674       A       1          OK          OK
  4      0            B    0.1859    0.8141            2    0.0833    0.9167       B       2          OK          OK
  5      0            A    0.9412    0.0588            1    0.9615    0.0385       A       1          OK          OK
  6      0            B    0.07

In [ ]:
!python "/content/drive/MyDrive/Colab_Notebooks/Independent_Study/scripts/extract_logits_text.py" \
      --format both \
      --resume \
      --out-dir "/content/drive/MyDrive/Colab_Notebooks/Independent_Study/results"


Loading tokenizer from Qwen/Qwen2-7B-Instruct ...
config.json: 100% 663/663 [00:00<00:00, 1.95MB/s]
tokenizer_config.json: 100% 1.29k/1.29k [00:00<00:00, 3.18MB/s]
vocab.json: 100% 2.78M/2.78M [00:00<00:00, 44.5MB/s]
merges.txt: 100% 1.67M/1.67M [00:00<00:00, 98.4MB/s]
tokenizer.json: 100% 7.03M/7.03M [00:00<00:00, 65.0MB/s]
Loading model in 4-bit ...
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
model.safetensors.index.json: 100% 27.8k/27.8k [00:00<00:00, 75.6MB/s]
Fetching 4 files: 100% 4/4 [00:41<00:00, 10.48s/it]
Download complete: 100% 15.2G/15.2G [00:41<00:00, 363MB/s]
Loading weights:   1% 2/339 [00:06<17:11,  3.06s/it]/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
Loading weights: 100% 339/339 [00:39<00:00,  8.60it/s]
generation_config.json: 100% 243/243 [00

In [ ]:
!python "/content/drive/MyDrive/Colab_Notebooks/Independent_Study/scripts/extract_cascade.py" \
      --phase both --resume \
      --out-dir "/content/drive/MyDrive/Colab_Notebooks/Independent_Study/results"


Loading dataset slprl/StressTest ...

=== Phase 1: Whisper transcribe → /content/drive/MyDrive/Colab_Notebooks/Independent_Study/results/cascade_transcripts.jsonl ===
[transcribe] resuming — 218 items already in /content/drive/MyDrive/Colab_Notebooks/Independent_Study/results/cascade_transcripts.jsonl
Loading Whisper from openai/whisper-large-v3 ...
preprocessor_config.json: 100% 340/340 [00:00<00:00, 1.21MB/s]
config.json: 100% 1.27k/1.27k [00:00<00:00, 4.41MB/s]
tokenizer_config.json: 100% 283k/283k [00:00<00:00, 213MB/s]
vocab.json: 100% 1.04M/1.04M [00:00<00:00, 26.3MB/s]
tokenizer.json: 100% 2.48M/2.48M [00:00<00:00, 101MB/s]
merges.txt: 100% 494k/494k [00:00<00:00, 62.3MB/s]
normalizer.json: 100% 52.7k/52.7k [00:00<00:00, 55.1MB/s]
added_tokens.json: 100% 34.6k/34.6k [00:00<00:00, 69.7MB/s]
special_tokens_map.json: 100% 2.07k/2.07k [00:00<00:00, 5.90MB/s]
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
model.safetensors: 100% 3.09G/3.09G [00:10<00:00, 299MB/s]
Lo

In [ ]:
!python "/content/drive/MyDrive/Colab_Notebooks/Independent_Study/scripts/noise_floor.py" \
      --inspect 5 \
      --out-dir "/content/drive/MyDrive/Colab_Notebooks/Independent_Study/results"


Loading dataset slprl/StressTest ...
Writing 5 clean+perturbed sample sets to /content/drive/MyDrive/Colab_Notebooks/Independent_Study/results/perturbed_samples ...
  idx=0  transcription="I didn't say he stole the money."
  idx=1  transcription="I didn't say he stole the money."
  idx=2  transcription="I didn't take your book."
  idx=3  transcription="I didn't take your book."
  idx=4  transcription="I didn't take your book."
Done. Listen to a few. If anything sounds meaning-changing (e.g. stress pattern altered, syllables dropped), revisit the perturbation constants at the top of this script.


In [ ]:
!python "/content/drive/MyDrive/Colab_Notebooks/Independent_Study/scripts/noise_floor.py" \
      --format both --resume \
      --out-dir "/content/drive/MyDrive/Colab_Notebooks/Independent_Study/results"


Loading dataset slprl/StressTest ...
Loading processor from Qwen/Qwen2-Audio-7B-Instruct ...
Loading model in 4-bit ...
Fetching 5 files: 100% 5/5 [00:00<00:00, 29873.96it/s]
Download complete: : 0.00B [00:00, ?B/s]
Loading weights:   0% 0/876 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
Loading weights: 100% 876/876 [01:00<00:00, 14.53it/s]
[transformers] The tied weights mapping and config for this model specifies to tie model.language_model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.

=== Noise floor: gain / Format A → /content/drive/MyDrive/Colab_Notebooks/Independent_Study/results/logits_noise_g

In [ ]:
#!python "/content/drive/MyDrive/Colab_Notebooks/Independent_Study/scripts/noise_floor.py" \
      --inspect 5 \
      --out-dir "/content/drive/MyDrive/Colab_Notebooks/Independent_Study/results"


Loading dataset slprl/StressTest ...
Writing 5 clean+perturbed sample sets to /content/drive/MyDrive/Colab_Notebooks/Independent_Study/results/perturbed_samples ...
  idx=0  transcription="I didn't say he stole the money."
  idx=1  transcription="I didn't say he stole the money."
  idx=2  transcription="I didn't take your book."
  idx=3  transcription="I didn't take your book."
  idx=4  transcription="I didn't take your book."
Done. Listen to a few. If anything sounds meaning-changing (e.g. stress pattern altered, syllables dropped), revisit the perturbation constants at the top of this script.


In [ ]:
#!python "/content/drive/MyDrive/Colab_Notebooks/Independent_Study/scripts/noise_floor.py" \
      --perturbations stretch_slow,stretch_fast \
      --format both \
      --out-dir "/content/drive/MyDrive/Colab_Notebooks/Independent_Study/results"


Loading dataset slprl/StressTest ...
Loading processor from Qwen/Qwen2-Audio-7B-Instruct ...
Loading model in 4-bit ...
Fetching 5 files: 100% 5/5 [00:00<00:00, 6448.81it/s]
Download complete: : 0.00B [00:00, ?B/s]
Loading weights:   0% 0/876 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
Loading weights: 100% 876/876 [00:42<00:00, 20.66it/s]
[transformers] The tied weights mapping and config for this model specifies to tie model.language_model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.

=== Noise floor: stretch_slow / Format A → /content/drive/MyDrive/Colab_Notebooks/Independent_Study/results/logits_